# 真实本地模型：FActScore 风格证据核验

用真实本地 NLI、`LocalCorpusRetriever` 与 `FActScoreScorer` 对输入回答产生 claim-level 结论。

## 真实集成前提

安装：`python3 -m pip install -e ".[nli]"`；打包证据语料仅覆盖教材事实。

代码单元没有预执行。填写 API 凭据或本地模型绝对路径后，从上到下运行；不要将 API key 保存到 notebook。

## 参数说明

- `LOCAL_NLI_MODEL: str`：本地 NLI 模型路径。
- `support_threshold: float = 0.5`：支持判定阈值。
- `contradiction_threshold: float = 0.5`：矛盾判定阈值。
- `evidence_per_claim: int = 5`：每 claim 的候选证据数。

## 步骤 1：配置真实 NLI

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from hallucination.detection.backends.nli import BidirectionalNLIBackend
LOCAL_NLI_MODEL = "/absolute/path/to/your/nli-model"
NLI_DEVICE = "cpu"
if not Path(LOCAL_NLI_MODEL).is_dir():
    raise ValueError("Set LOCAL_NLI_MODEL to an existing local NLI model directory.")
nli = BidirectionalNLIBackend(LOCAL_NLI_MODEL, device=NLI_DEVICE)

## 步骤 2：建立证据管线

In [ ]:
from hallucination.detection.evidence import FActScoreScorer, LocalCorpusRetriever, NLIEvidenceVerifier, load_textbook_evidence
from hallucination.detection.pipelines import EvidenceVerificationPipeline
pipeline = EvidenceVerificationPipeline(FActScoreScorer(NLIEvidenceVerifier(nli, support_threshold=0.5, contradiction_threshold=0.5), retriever=LocalCorpusRetriever(load_textbook_evidence()), evidence_per_claim=5))

## 步骤 3：输入回答

In [ ]:
response = input("回答文本：").strip()
claims_text = input("可选 claims，用 | 分隔；直接回车则按句切分：").strip()
claims = [item.strip() for item in claims_text.split("|") if item.strip()] or None
result = pipeline.detect(response=response, claims=claims)
for claim in result.claims:
    print(claim.label, claim.claim, claim.score, [(e.title, e.snippet) for e in claim.evidence])
print(result.method_scores[0].raw)

## 结果解读与限制

`unknown` 不等于 hallucination；它表示当前检索和 NLI 无法定论。生产部署必须替换为受控且可追溯的知识语料。